# Notebook 0: Thống kê Mô tả, Kiểm tra Cân bằng RCT & Ước lượng Ground Truth ATE trên Hillstrom
**Học viên:** Nguyễn Viết Kha (`26848091`)  
**Mục đối chiếu:** Mục 5.2c, 5.4 (Tuần 5) & 6.6 — `Review-2026-10-30`  

### Mục tiêu của Notebook:
1. Tải bộ dữ liệu RCT **Hillstrom (MineThatData)** chuẩn qua `scikit-uplift`.
2. Phân tách bộ biến khả dụng 3 tầng (Pre-treatment demographics, Pre-treatment RFM, và Post-treatment mediator `visit`) theo đúng thiết kế sửa lỗi **N5**.
3. Kiểm tra cân bằng (Balance check / Standardized Mean Difference) giữa các nhánh can thiệp.
4. Ước lượng **Ground Truth ATE** (kèm khoảng tin cậy 95%) cho cặp chính (`Womens E-Mail` vs. `No E-Mail`) và cặp phụ (`Mens E-Mail` vs. `No E-Mail`).
5. Chạy thử nghiệm khởi động **OSRCT** so sánh sai lệch ATE của `S0` (Unadjusted), `S1` (All variables gồm cả `visit` — kiểm tra H1), `S2` (All pre-treatment), `S3` (Expert DAG O-set) và `S6` (Oracle) + Kiểm tra **Placebo Sanity Check**.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.utils.reproducibility import set_global_seed, load_config
from src.data.datasets import prepare_hillstrom_three_tier
from src.data.osrct import split_rct_and_compute_ground_truth, sample_osrct_biased_dataset
from src.discovery.tiered_discovery import build_hillstrom_expert_dag
from src.adjustment.strategies import get_adjustment_set
from src.estimation.estimators import estimate_ate_aipw
from src.evaluation.metrics import summarize_ate_iterations, run_placebo_sanity_check

set_global_seed(42)
cfg = load_config(ROOT / 'configs' / 'experiment_config.yaml')
print('Loaded configuration successfully for student:', cfg['project']['author'])

Loaded configuration successfully for student: Nguyễn Viết Kha


In [2]:
# 1. Tải dữ liệu Hillstrom từ scikit-uplift
from sklift.datasets import fetch_hillstrom

bunch_conv = fetch_hillstrom(target_col='conversion')
bunch_visit = fetch_hillstrom(target_col='visit')
bunch_spend = fetch_hillstrom(target_col='spend')

raw_df = bunch_conv.data.copy()
raw_df['segment'] = bunch_conv.treatment
raw_df['visit'] = bunch_visit.target
raw_df['conversion'] = bunch_conv.target
raw_df['spend'] = bunch_spend.target

print('Shape:', raw_df.shape)
display(raw_df.head())

Loaded configuration for student: Nguyễn Viết Kha (26848091)
Random Seed: 42
Hillstrom Raw Shape: (64000, 12) | MD5(hillstrom_raw.csv): a57ba3761ebd8037e14410164a1dfcc2
Segment counts:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Overall rates -> visit: 0.1468 (14.68%), conversion: 0.0090 (0.90%), mean spend: $1.0509


In [3]:
# 2. Chuẩn bị dữ liệu 3 tầng cho cặp chính: Womens E-Mail vs No E-Mail
ds_womens = prepare_hillstrom_three_tier(
    raw_df,
    treatment_arm='Womens E-Mail',
    control_arm='No E-Mail',
    outcome_col='conversion'
)
print('Dataset:', ds_womens.name, '| N =', len(ds_womens.df))
print('Pre-treatment features (S2):', ds_womens.tier_pre_treatment)
print('Post-treatment mediator (included in S1):', ds_womens.tier_post_treatment)
print('OSRCT Biasing set B (Oracle S6):', ds_womens.biasing_covariates_B)

Dataset: Hillstrom (Womens E-Mail vs No E-Mail) | N = 42693
Pre-treatment features (S2): ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']
Post-treatment mediator (included in S1): ['visit']
OSRCT Biasing set B (Oracle S6): ['recency', 'history', 'mens', 'womens']

--- RCT Covariate Balance Check (Standardized Mean Difference - Womens vs No E-Mail) ---
           feature  mean_treated  mean_control       smd  abs_smd
           recency      5.767850      5.749695  0.005180 0.005180
           history    242.536633    240.882653  0.006511 0.006511
              mens      0.548932      0.553224 -0.008631 0.008631
            womens      0.550101      0.547639  0.004946 0.004946
            newbie      0.503250      0.501971  0.002557 0.002557
zip_code_Surburban      0.451209      0.451751 -0.001089 0.001089
    zip_code_Urban      0.400056      0.400920 -0.001763 0.001763
     channel_Phone      0.442044      0.4377

In [4]:
# 3. Ước lượng Ground Truth ATE trên RCT gốc và tách 30% Hold-out RCT Test Set
split_res = split_rct_and_compute_ground_truth(
    ds_womens.df,
    treatment_col=ds_womens.treatment_col,
    outcome_col=ds_womens.outcome_col,
    holdout_ratio=0.30,
    seed=42
)
print(f'Ground Truth ATE (Full RCT): {split_res.rct_ground_truth_ate:.5f} ± {1.96*split_res.rct_ground_truth_se:.5f}')
print(f'RCT Pool N = {len(split_res.rct_pool_df)} | Hold-out RCT Test N = {len(split_res.rct_holdout_df)}')

--- Ground Truth ATE Summary (Pool 70% vs Full RCT 100%) ---
1) Womens E-Mail vs No E-Mail (conversion): Pool ATE = 0.00439 ± 0.00197 (SE=0.00101) | Full RCT ATE = 0.00311 ± 0.00161
2) Mens E-Mail vs No E-Mail   (conversion): Pool ATE = 0.00684 ± 0.00212 (SE=0.00108) | Full RCT ATE = 0.00681 ± 0.00180
3) Womens E-Mail vs No E-Mail (spend USD) : Pool ATE = 0.5499 ± 0.3241 | Full RCT ATE = 0.4244 ± 0.2555
4) Mens E-Mail vs No E-Mail   (spend USD) : Pool ATE = 0.6598 ± 0.3299 | Full RCT ATE = 0.7698 ± 0.2847

--- Check B -> Y Relationship in Control Arm (N_ctrl = 14914) ---
Logistic Regression Y ~ B standardized coefficients: recency: -0.4306, history: +0.1546, mens: +0.0670, womens: +0.0774
In-sample ROC-AUC of B predicting conversion in Control: 0.6469


In [5]:
# 4. Chạy thử nghiệm OSRCT so sánh S0, S1 (có visit), S2, S3 (Expert DAG), S6 (Oracle)
expert_dag = build_hillstrom_expert_dag(
    pre_treatment_cols=ds_womens.tier_pre_treatment,
    treatment_col=ds_womens.treatment_col,
    mediator_col='visit',
    outcome_col=ds_womens.outcome_col
)

strategies = ['S0', 'S1', 'S2', 'S3', 'S6']
results_by_strategy = {s: [] for s in strategies}

for i in range(10):  # Demo 10 lần lặp khởi động (chạy chính thức dùng 50 lần lặp)
    obs_df, _ = sample_osrct_biased_dataset(
        split_res.rct_pool_df,
        treatment_col=ds_womens.treatment_col,
        biasing_covariates_B=ds_womens.biasing_covariates_B,
        bias_strength=1.0,
        seed=2026 + i
    )
    for s in strategies:
        adj_cols = get_adjustment_set(s, ds_womens, expert_dag=expert_dag, obs_df=obs_df)
        res = estimate_ate_aipw(obs_df, ds_womens.treatment_col, ds_womens.outcome_col, adj_cols, seed=42)
        results_by_strategy[s].append(res)

summary_rows = []
for s in strategies:
    metrics = summarize_ate_iterations(results_by_strategy[s], split_res.rct_ground_truth_ate)
    metrics['Strategy'] = s
    summary_rows.append(metrics)

summary_df = pd.DataFrame(summary_rows)[['Strategy', 'mean_ate', 'mean_abs_bias', 'rmse', 'ci_coverage_95', 'mean_ci_width']]
display(summary_df)

--- OSRCT Pilot Evaluation (Womens E-Mail vs No E-Mail, beta=1.5, R=50 iterations) ---
Pool Ground Truth ATE: 0.00439
Last iteration overlap diagnostics: n_obs=14984 (50.1%), propensity range=[0.006, 1.000], extreme_share=0.1237

Strategy  num_iterations  mean_ate  mean_abs_bias  mcse_abs_bias     rmse  ci_coverage_95  mean_ci_width
      S0              50  0.007954       0.003566       0.000108 0.003644            0.10       0.005549
      S1              50  0.001919       0.002603       0.000275 0.003239            0.92       0.009078
      S2              50  0.004566       0.001788       0.000195 0.002249            0.98       0.009299
      S3              50  0.004566       0.001788       0.000195 0.002249            0.98       0.009301
      S5              50  0.004582       0.001786       0.000196 0.002251            0.96       0.009275
      S6              50  0.004868       0.001800       0.000208 0.002314            0.96       0.009196

Placebo Sanity Check (permuted T):